In [1]:
import torch
import vae_model
import optuna

In [2]:
bounds = torch.tensor([[1 , 1 , 1 , 30] ,
                       [60, 60, 60, 50]],dtype = torch.float64)
num_initial_points = 5000
initial_points = vae_model.data_encoder_generation(bounds , num_initial_points=num_initial_points)
print('Number of points generated:', len(initial_points))

Number of points generated: 5000


Split data

In [3]:
train_dataloader , test_dataloader = vae_model.split_data(initial_points)

Optimizacion con hyperparametros

In [4]:
# Crear estudio
study_name = 'Test'
storage_traj = f'sqlite:///{study_name}.db'
study = optuna.create_study(direction='minimize', study_name=study_name, storage=storage_traj, load_if_exists=True)


[I 2025-12-07 07:57:16,475] Using an existing study with name 'Test' instead of creating a new one.


In [5]:
trials = 1 # set the number of trials that you want to run , you can re-run any times. 
columns_to_encode = initial_points
X_discrete_raw = columns_to_encode.clone()  
for i in range(trials):
    study.optimize(lambda trial: vae_model.objective(trial, X_discrete_raw), n_trials=1)

100%|██████████| 250/250 [03:21<00:00,  1.24it/s]
[I 2025-12-07 08:16:51,875] Trial 1 finished with value: 3.060974292755127 and parameters: {'Latent size': 8, 'Emb size': 34, 'Hidden1': 120, 'KL ratio': 0.12235129129160766, 'lr': 0.00014077851335300207}. Best is trial 1 with value: 3.060974292755127.


In [ ]:
# from optuna.visualization import plot_optimization_history, plot_param_importances, plot_slice
# plot_optimization_history(study)
# plot_param_importances(study)
# plot_slice(study)

In [ ]:
## uncomment just in case the study must be deleted.
# optuna.delete_study(study_name=study_name, storage=storage_traj)

In [ ]:
# study.best_params

In [ ]:
parameters =  {'Latent size': 7, 
               'Emb size': 12, 
               'Hidden1': 88, 
               'KL ratio': 0.1557071535379246, 
               'lr': 0.0005582408993581889}

In [ ]:
# Function doesnt save the .pth file (final model) because those lines are marked as comments 
model = vae_model.train_model(train_dataloader,test_dataloader,parameters)

In [ ]:
discrete_values = vae_model.re_evaluate(initial_points, model)